In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo
from matplotlib import rc
from cartopy.crs import Mercator, PlateCarree
from matplotlib.patches import Rectangle

from auxdata import get_multibeam_map_W1, get_multibeam_map_W3
from plot import nice_lonlat_gridlines, scale_bar

rc('font', size=8)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 4.5

Multibeam maps

In [ ]:
W1 = get_multibeam_map_W1()
W3 = get_multibeam_map_W3()

ADCP missions

In [ ]:
mission_02 = xr.open_dataset('data/derived/NBP2202_02_cleaned.nc')
mission_03 = xr.open_dataset('data/derived/NBP2202_03_cleaned.nc')
mission_04 = xr.open_dataset('data/derived/NBP2202_04_cleaned.nc')

Background

In [ ]:
sar = xr.load_dataset('data/auxiliary/background/2022-01-21-00_00_2022-01-21-23_59_Sentinel-1_IW_HH_HH_-_decibel_gamma0.nc')
sar

### Make figure

In [ ]:
lon_min = -113.8
lon_max = -111.3
lat_max = -73.95
lat_min = -74.35

ADCP02_color = 'dodgerblue'
ADCP03_color = 'deeppink'
ADCP04_color = 'goldenrod'
W1_color = 'green'
W3_color = 'darkorange'

plt.figure(figsize=(fig_width, fig_height), layout='tight')
ax = plt.axes(projection = Mercator(central_longitude=-112.75,
                                    min_latitude = -75,
                                    max_latitude = -73,
                                    latitude_true_scale = -74.2))


# Plot SAR background
ax.pcolormesh(sar.lon, sar.lat, sar.Band1, cmap=cmo.gray, transform = PlateCarree())

step = 10
# Plot AUV missions
ax.plot(mission_02.longitude[::step], mission_02.latitude[::step], c=ADCP02_color, transform=PlateCarree(), linewidth=0.7)
ax.plot(mission_03.longitude[::step], mission_03.latitude[::step], c=ADCP03_color, transform=PlateCarree(), linewidth=0.7)
ax.plot(mission_04.longitude[::step], mission_04.latitude[::step], c=ADCP04_color, transform=PlateCarree(), linewidth=0.7)

# Plot multibeam reference
ax.scatter(W3.Lon[::step], W3.Lat[::step], transform=PlateCarree(), s=1, alpha = 0.02, color=W3_color)
ax.scatter(W1.Lon[::step], W1.Lat[::step], transform=PlateCarree(), s=1, alpha = 0.02, color=W1_color)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=PlateCarree())
gl = nice_lonlat_gridlines(ax, size=7, latitudes = [-74.3, -74.2, -74.1, -74],zorder=10, alpha=0.2)
scale_bar(ax, length=5, location = (0.03,0.03), textoffset=300, fontsize=8, linewidth=2)


# Legend
ax.plot([1,2],[1,2], label='Mission 02', color = ADCP02_color, zorder=-10)
ax.plot([1,2],[1,2], label='Mission 03', color = ADCP03_color, zorder=-10)
ax.plot([1,2],[1,2], label='Mission 04', color = ADCP04_color, zorder=-10)
ax.add_patch(Rectangle((0,0),1,1, facecolor = W1_color, edgecolor = None, alpha = 0.8, zorder=-10, label='W1'))
ax.add_patch(Rectangle((0,0),1,1, facecolor = W3_color, edgecolor = None, alpha = 0.8, zorder=-10, label='W3'))
plt.legend(loc='lower right', fontsize=7)

plt.savefig('figures/fig2.png', dpi=400, bbox_inches='tight')